# 07 -- Проверка простых изменений

После первой отправки получил **0.821574** (результат отправки, разметка недоступна).
Проверяю две гипотезы: можно ли сократить пул перед CatBoost и помогает ли примесь RRF.
Использую только dev и модель, обученную на rank_train. Финальная refit-модель сюда не подходит: она видела dev.
Holdout уже открыт. Новые выводы считаю дополнительными dev-экспериментами, а не независимой оценкой.
Исходный answer.csv не меняю.

In [1]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display

ex_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
get_ipython().run_line_magic('run', str(ex_root / 'notebooks/common.ipynb'))

ex_out = ex_root / 'outputs'
ex_queries = pd.read_parquet(ex_out / 'cache/eval_queries.parquet').query(
    "split == 'dev'"
)
ex_all_truth = json.loads((ex_out / 'cache/truth.json').read_text())
ex_truth = {q: ex_all_truth[q] for q in ex_queries.query_id}
ex_items = pd.read_parquet(ex_out / 'cache/items.parquet', columns=['item_id'])
ex_frame = pd.read_parquet(ex_out / 'cache/fusion_dev.parquet')
ex_config = json.loads((ex_out / 'models/selected.json').read_text())
ex_columns = json.loads((ex_out / 'models/fusion_features.json').read_text())[
    'features'
]
ex_warm_share = json.loads((ex_out / 'models/hybrid.json').read_text())['warm_share']
ex_model = CatBoostClassifier().load_model(str(ex_out / 'models/fusion.cbm'))
ex_frame = ex_frame.sort_values(
    ['query_id', 'rrf', 'pos'], ascending=[True, False, True]
).reset_index(drop=True)


In [2]:
ex_frame['pool_rank'] = ex_frame.groupby('query_id').cumcount() + 1
ex_start = time.perf_counter()
ex_frame['model_score'] = ex_model.predict_proba(
    model_matrix(ex_frame, ex_columns), ntree_end=ex_config['ntree']
)[:, 1]
print(
    'Dev rows:',
    len(ex_frame),
    'scoring seconds:',
    round(time.perf_counter() - ex_start, 2),
)
ex_frame = ex_frame.sort_values(
    ['query_id', 'model_score', 'pos'], ascending=[True, False, True]
)
ex_frame['model_rank'] = ex_frame.groupby('query_id').cumcount() + 1
ex_baseline = recall_table(
    predictions_from_frame(ex_frame, ex_items, score='model_score'), ex_truth
).set_index('query_id')
ex_baseline = ex_baseline.join(ex_queries.set_index('query_id')[['regime']])


Dev rows: 784839 scoring seconds: 0.09


## Размер пула и смешивание рангов

Для пула оставляю первые 50/100/200/300/400 кандидатов по исходному RRF, затем применяю тот же CatBoost.
Это проверка сокращения инференса без переобучения. Признаки и исходный n_sources сохраняю.
Время поиска и построения признаков здесь не измеряю, поэтому долю оставшихся строк не выдаю за ускорение всей системы.

Для смешивания использую `(1 - alpha) / (30 + model_rank) + alpha / (30 + rrf_rank)`.
Так не приходится сравнивать шкалу вероятностей CatBoost со шкалой RRF.
Парный bootstrap сохраняет warm/cold и сравнивает каждый запрос с исходной моделью.
Интервалы описательные: dev уже использован для выбора, поправки на перебор нет.

In [3]:
ex_variants = [(f'pool_{k}', k, 0.0) for k in [50, 100, 200, 300, 400]]
ex_variants += [(f'blend_{a:g}', 400, a) for a in [0.05, 0.1, 0.2, 0.35, 0.5, 1.0]]
ex_rows = []
ex_details = []
for ex_name, ex_pool, ex_alpha in ex_variants:
    ex_part = ex_frame[ex_frame.pool_rank.le(ex_pool)].copy()
    ex_part['score'] = (1 - ex_alpha) / (30 + ex_part.model_rank) + ex_alpha / (
        30 + ex_part.pool_rank
    )
    ex_metrics = recall_table(
        predictions_from_frame(ex_part, ex_items, score='score'), ex_truth
    ).set_index('query_id')
    ex_metrics = ex_metrics.join(ex_baseline[['regime']])
    ex_metrics['delta'] = ex_metrics['recall@50'] - ex_baseline['recall@50']
    ex_scores = ex_metrics.groupby('regime')['recall@50'].mean()
    ex_rng = np.random.default_rng(143)
    ex_boot = np.zeros(2000)
    for ex_regime, ex_weight in [('warm', ex_warm_share), ('cold', 1 - ex_warm_share)]:
        ex_delta = ex_metrics.loc[ex_metrics.regime.eq(ex_regime), 'delta'].to_numpy()
        ex_boot += ex_weight * ex_rng.choice(
            ex_delta, (2000, len(ex_delta)), replace=True
        ).mean(axis=1)
    ex_ci = np.quantile(ex_boot, [0.025, 0.975])
    ex_rows.append(
        {
            'variant': ex_name,
            'pool': ex_pool,
            'alpha': ex_alpha,
            'cold': ex_scores['cold'],
            'warm': ex_scores['warm'],
            'objective': (1 - ex_warm_share) * ex_scores['cold']
            + ex_warm_share * ex_scores['warm'],
            'delta_ci_low': ex_ci[0],
            'delta_ci_high': ex_ci[1],
            'candidate_rows': len(ex_part),
            'improved_queries': int(ex_metrics.delta.gt(0).sum()),
            'worse_queries': int(ex_metrics.delta.lt(0).sum()),
        }
    )
    ex_details.append(ex_metrics.reset_index().assign(variant=ex_name))


In [4]:
ex_results = pd.DataFrame(ex_rows)
ex_results.to_csv(ex_out / 'validation/additional_dev_experiments.csv', index=False)
pd.concat(ex_details, ignore_index=True).to_parquet(
    ex_out / 'validation/additional_dev_per_query.parquet', index=False
)
display(ex_results.sort_values('objective', ascending=False))


,variant,pool,alpha,cold,warm,objective,delta_ci_low,delta_ci_high,candidate_rows,improved_queries,worse_queries
5,blend_0.05,400,0.05,0.803139,0.812536,0.806661,0.000562,0.005250,784839,8,1
6,blend_0.1,400,0.10,0.803972,0.809036,0.805870,-0.001375,0.005419,784839,10,6
7,blend_0.2,400,0.20,0.801806,0.808036,0.804141,-0.004813,0.005232,784839,15,15
4,pool_400,400,0.00,0.800639,0.809536,0.803973,0.000000,0.000000,784839,0,0
3,pool_300,300,0.00,0.800806,0.806536,0.802953,-0.004645,0.002461,596844,6,8
8,blend_0.35,400,0.35,0.795306,0.803679,0.798444,-0.012406,0.000858,784839,22,32
9,blend_0.5,400,0.50,0.788306,0.798571,0.792153,-0.020472,-0.003170,784839,31,56
2,pool_200,200,0.00,0.784806,0.802286,0.791357,-0.019514,-0.006448,399854,13,37
1,pool_100,100,0.00,0.765806,0.790452,0.775043,-0.038901,-0.019544,200000,26,83
0,pool_50,50,0.00,0.736014,0.759988,0.744999,-0.071795,-0.046705,100000,39,159


## Решение

Ориентир -- исходный `pool_400`. Учитываю оба режима и парную разницу, а не только лучшее число в таблице.
Небольшой выигрыш на многократно использованном dev не достаточен для новой отправки.
Если сокращение пула ухудшает полноту, оставляю 400: простота не должна снижать качество.
Все варианты этого ноутбука -- локальные эксперименты. Новых официальных результатов пока нет.

## Кандидат для второй отправки

Примесь RRF 5% дала 0.80666 против 0.80397 на dev: улучшились 8 запросов, ухудшился 1.
Это небольшой результат на уже использованной выборке. Сохраняю отдельный файл, исходный ответ оставляю.
Перед смешиванием проверяю, что повторный расчет исходной модели точно восстанавливает отправленный ответ.
Параметр 5% выбран по dev; официальную разметку для него не использую.

In [5]:
ex_train_raw, ex_bq_raw, ex_bi_raw = load_raw()
ex_bq = prepare_queries(ex_bq_raw)
ex_catalog = pd.read_parquet(ex_out / 'cache/items.parquet')
ex_numeric = pd.read_parquet(ex_out / 'cache/item_features.parquet')
ex_positions = np.flatnonzero(ex_catalog.item_id.isin(ex_bi_raw.item_id).to_numpy())
ex_bitems = ex_catalog.iloc[ex_positions].reset_index(drop=True)
ex_bnumeric = ex_numeric.iloc[ex_positions].reset_index(drop=True)
ex_centers = ex_catalog.groupby('item_location_id')[
    ['item_latitude', 'item_longitude']
].median()
ex_pairs = prepare_queries(ex_train_raw)
ex_pairs['item_id'] = ex_train_raw.item_id.to_numpy()
ex_pairs = ex_pairs.drop_duplicates(['context_key', 'item_id'])
ex_counts = ex_pairs.item_id.value_counts().to_dict()
ex_embedding_ids = pd.read_parquet(
    ex_out / 'cache/embedding_query_ids.parquet'
).query_id.tolist()
ex_vectors = dict(zip(ex_embedding_ids, np.load(ex_out / 'cache/query_embeddings.npy')))
ex_ivectors = np.asarray(
    np.load(ex_out / 'cache/item_embeddings.npy', mmap_mode='r')[ex_positions]
)
ex_candidates = pd.read_parquet(ex_out / 'cache/benchmark_candidates.parquet')
ex_bfeatures = candidate_features(
    ex_candidates,
    ex_bq,
    ex_bitems,
    ex_bnumeric,
    ex_counts,
    ex_config['weights'],
    ex_vectors,
    ex_ivectors,
    max_candidates=400,
    centers=ex_centers,
)


In [6]:
ex_final = CatBoostClassifier().load_model(str(ex_out / 'models/fusion_final.cbm'))
ex_bfeatures['model_score'] = ex_final.predict_proba(
    model_matrix(ex_bfeatures, ex_columns)
)[:, 1]
ex_original = (
    pd.read_csv(ex_out / 'submissions/answer.csv', dtype=str, keep_default_na=False)
    .set_index('query_id')
    .answer.to_dict()
)
ex_pop = sorted(ex_bitems.item_id, key=lambda i: (-ex_counts.get(i, 0), i))[:100]
ex_restored = predictions_from_frame(ex_bfeatures, ex_bitems, score='model_score', k=50)
for ex_qid in ex_bq.query_id:
    ex_top = ex_restored.get(ex_qid, [])
    ex_restored[ex_qid] = (ex_top + [i for i in ex_pop if i not in set(ex_top)])[:50]
assert {
    q: ' '.join(v) for q, v in ex_restored.items()
} == ex_original, 'Исходный ответ не восстановлен'
ex_bfeatures = ex_bfeatures.sort_values(
    ['query_id', 'model_score', 'pos'], ascending=[True, False, True]
)
ex_bfeatures['model_rank'] = ex_bfeatures.groupby('query_id').cumcount() + 1
ex_bfeatures = ex_bfeatures.sort_values(
    ['query_id', 'rrf', 'pos'], ascending=[True, False, True]
)
ex_bfeatures['rrf_rank'] = ex_bfeatures.groupby('query_id').cumcount() + 1
ex_bfeatures['score'] = 0.95 / (30 + ex_bfeatures.model_rank) + 0.05 / (
    30 + ex_bfeatures.rrf_rank
)


In [7]:
ex_predictions = predictions_from_frame(ex_bfeatures, ex_bitems, score='score', k=50)
for ex_qid in ex_bq.query_id:
    ex_top = ex_predictions.get(ex_qid, [])
    ex_predictions[ex_qid] = (ex_top + [i for i in ex_pop if i not in set(ex_top)])[:50]
ex_path = ex_out / 'submissions/answer_blend005.csv'
ex_validation = save_submission(ex_predictions, ex_bq, ex_bitems, ex_path)
ex_validation['baseline_exactly_reproduced'] = True
ex_validation['queries_with_changed_set'] = sum(
    set(ex_predictions[q]) != set(ex_original[q].split()) for q in ex_bq.query_id
)
ex_validation['replaced_items'] = sum(
    len(set(ex_predictions[q]) - set(ex_original[q].split())) for q in ex_bq.query_id
)
ex_validation['sha256'] = sha256_file(ex_path)
ex_validation['official_score'] = None
write_json(ex_validation, ex_out / 'submissions/validation_blend005.json')
display(ex_validation)


{'rows': 2452,
 'min_candidates': 50,
 'max_candidates': 50,
 'valid': True,
 'baseline_exactly_reproduced': True,
 'queries_with_changed_set': 1357,
 'replaced_items': 1493,
 'sha256': 'b35501639ce7f471fa5b35119437c015b49d52d99e3ae8b465466a06b8754bea',
 'official_score': None}